## Step 1: Import Necessary Libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)

## Step 2: Load all 9 csv files

In [2]:
customers   = pd.read_csv('Brazilian E-Commerce Analytics Dataset/olist_customers_dataset.csv')
orders      = pd.read_csv('Brazilian E-Commerce Analytics Dataset/olist_orders_dataset.csv')
order_items = pd.read_csv('Brazilian E-Commerce Analytics Dataset/olist_order_items_dataset.csv')
payments    = pd.read_csv('Brazilian E-Commerce Analytics Dataset/olist_order_payments_dataset.csv')
reviews     = pd.read_csv('Brazilian E-Commerce Analytics Dataset/olist_order_reviews_dataset.csv')
products    = pd.read_csv('Brazilian E-Commerce Analytics Dataset/olist_products_dataset.csv')
sellers     = pd.read_csv('Brazilian E-Commerce Analytics Dataset/olist_sellers_dataset.csv')
geolocation = pd.read_csv('Brazilian E-Commerce Analytics Dataset/olist_geolocation_dataset.csv')
cat_translation = pd.read_csv('Brazilian E-Commerce Analytics Dataset/product_category_name_translation.csv')

tables = {
    'customers': customers, 'orders': orders, 'order_items': order_items,
    'payments': payments, 'reviews': reviews, 'products': products,
    'sellers': sellers, 'geolocation': geolocation, 'cat_translation': cat_translation
}

for name, data in tables.items():
    print(f"{name:15s} shape={data.shape}")


customers       shape=(99441, 5)
orders          shape=(99441, 8)
order_items     shape=(112650, 7)
payments        shape=(103886, 5)
reviews         shape=(99224, 7)
products        shape=(32951, 9)
sellers         shape=(3095, 4)
geolocation     shape=(1000163, 5)
cat_translation shape=(71, 2)


## Step 3: Inspect structures, dtypes and nulls

In [3]:
for table_name, dataset in tables.items():

    print(f"\n{'='*60}")
    print(table_name)
    print('='*60)

    print(dataset.dtypes)

    print("\nNull counts:")
    null_counts = dataset.isnull().sum()

    print(null_counts[null_counts > 0])


customers
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Null counts:
Series([], dtype: int64)

orders
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Null counts:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

order_items
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

Null counts:
Series([], dtype: int64)

payments
order_id      

## Step 4: Fix data types - change all date columns into datetime

In [4]:
date_cols_orders = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for c in date_cols_orders:
    orders[c] = pd.to_datetime(orders[c], errors='coerce')

reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'], errors='coerce')
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')

orders.dtypes[date_cols_orders]

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## Step 5: Handle Missing Values - Documented Decisions 

In [5]:
# Orders: keep delivery-date nulls as-is (order_status explains them)
print(orders['order_status'].value_counts())
print("\nNull delivered_customer_date by status:")
print(orders[orders['order_delivered_customer_date'].isnull()]['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Null delivered_customer_date by status:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [6]:
# Products: fill missing category with 'unknown'.
products['product_category_name'] = products['product_category_name'].fillna('unknown')

print("Products null count after category fill:")
print(products.isnull().sum()[products.isnull().sum() > 0])
print("Check Null values on payments")
print(payments.isnull().sum())

Products null count after category fill:
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
Check Null values on payments
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64


## Step 6: Add English category names(translation tables)

In [7]:
products = products.merge(cat_translation, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')
products[['product_category_name', 'product_category_name_english']].head()

,product_category_name,product_category_name_english
0,perfumaria,perfumery
1,artes,art
2,esporte_lazer,sports_leisure
3,bebes,baby
4,utilidades_domesticas,housewares


## Step 7: Merge into one master table 

In [8]:
# Aggregate payments per order (an order can have multiple payment installments/methods)
payments_agg = payments.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'max'),
    payment_type=('payment_type', lambda x: x.mode()[0] if not x.mode().empty else np.nan)
).reset_index()

# Aggregate reviews per order (take the latest review if duplicates exist)
reviews_agg = (reviews.sort_values('review_creation_date')
               .groupby('order_id')
               .agg(review_score=('review_score', 'last'))
               .reset_index())

In [9]:
master = (order_items
    .merge(orders, on='order_id', how='left')
    .merge(products, on='product_id', how='left')
    .merge(sellers, on='seller_id', how='left')
    .merge(customers, on='customer_id', how='left')
    .merge(payments_agg, on='order_id', how='left')
    .merge(reviews_agg, on='order_id', how='left')
)

print(master.shape)
master.head()

(112650, 34)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_value,payment_installments,payment_type,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,27277,volta redonda,SP,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,72.19,2.0,credit_card,5.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,3471,sao paulo,SP,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,259.83,3.0,credit_card,4.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,37564,borda da mata,MG,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,216.87,5.0,credit_card,5.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,14403,franca,SP,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP,25.78,2.0,credit_card,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,87900,loanda,PR,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP,218.04,3.0,credit_card,5.0


## 7. Rows checked before saving

In [10]:
# Row count check: should match order_items row count (one row per item, not inflated)
print("order_items rows:", len(order_items))
print("master rows:", len(master))
assert len(master) == len(order_items), "Row count mismatch — check for duplicate joins!"

# Revenue sanity check
print("\nTotal item price + freight (order_items):", (order_items['price'] + order_items['freight_value']).sum())
print("Null counts in master (top 10):")
print(master.isnull().sum().sort_values(ascending=False).head(10))

order_items rows: 112650
master rows: 112650

Total item price + freight (order_items): 15843553.24
Null counts in master (top 10):
order_delivered_customer_date    2454
product_name_lenght              1603
product_photos_qty               1603
product_description_lenght       1603
order_delivered_carrier_date     1194
review_score                      942
product_length_cm                  18
product_width_cm                   18
product_weight_g                   18
product_height_cm                  18
dtype: int64


In [11]:
# Add a delivery delay column
master['delivery_delay_days'] = (
    master['order_delivered_customer_date'] - master['order_estimated_delivery_date']
).dt.days

master[['order_id', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days']].head()

,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-20 23:43:48,2017-09-29,-9.0
1,00018f77f2f0320c557190d7a144bdd3,2017-05-12 16:04:24,2017-05-15,-3.0
2,000229ec398224ef6ca0657da4fc703e,2018-01-22 13:19:16,2018-02-05,-14.0
3,00024acbcdf0a6daa1e931b038114c75,2018-08-14 13:32:39,2018-08-20,-6.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,2017-03-01 16:42:31,2017-03-17,-16.0


In [12]:
master.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_value,payment_installments,payment_type,review_score,delivery_delay_days
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,27277,volta redonda,SP,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,72.19,2.0,credit_card,5.0,-9.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,3471,sao paulo,SP,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,259.83,3.0,credit_card,4.0,-3.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,37564,borda da mata,MG,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,216.87,5.0,credit_card,5.0,-14.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,14403,franca,SP,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP,25.78,2.0,credit_card,4.0,-6.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,87900,loanda,PR,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP,218.04,3.0,credit_card,5.0,-16.0


In [14]:
len(master)

112650

In [17]:
total_revenue = (master['price']+master['freight_value']).sum()
total_revenue

np.float64(15843553.24)

In [19]:
Distinct_order_count = master['order_id'].nunique()
Distinct_order_count

98666

## 8. Saved Master Table

In [13]:
master.to_csv('olist_master.csv', index=False)
print("Saved olist_master.csv —", master.shape)

PermissionError: [Errno 13] Permission denied: 'olist_master.csv'